In [12]:
!pip install mlflow dagshub -q

import os
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
os.environ["MLFLOW_TRACKING_PASSWORD"] = user_secrets.get_secret("DAGSHUB_TOKEN")
os.environ["MLFLOW_TRACKING_USERNAME"] = user_secrets.get_secret("DAGSHUB_USERNAME")

import mlflow
import mlflow.sklearn
mlflow.set_tracking_uri("https://dagshub.com/llikl23/house-prices-mlflow.mlflow")
print("MLflow connected!")

MLflow connected!


In [13]:
import pandas as pd
import numpy as np

DATA_DIR = "/kaggle/input/competitions/house-prices-advanced-regression-techniques"
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")

print(f"Train: {train.shape}")
print(f"Test:  {test.shape}")

Train: (1460, 81)
Test:  (1459, 80)


In [14]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split

none_cols = [
    "PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu",
    "GarageType", "GarageFinish", "GarageQual", "GarageCond",
    "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
    "MasVnrType",
]
zero_cols = ["GarageYrBlt", "MasVnrArea"]

quality_map = {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}
ordinal_maps = {
    "ExterQual": quality_map, "ExterCond": quality_map,
    "BsmtQual": quality_map, "BsmtCond": quality_map,
    "HeatingQC": quality_map, "KitchenQual": quality_map,
    "FireplaceQu": quality_map, "GarageQual": quality_map,
    "GarageCond": quality_map, "PoolQC": quality_map,
    "BsmtExposure": {"None": 0, "No": 1, "Mn": 2, "Av": 3, "Gd": 4},
    "BsmtFinType1": {"None": 0, "Unf": 1, "LwQ": 2, "Rec": 3, "BLQ": 4, "ALQ": 5, "GLQ": 6},
    "BsmtFinType2": {"None": 0, "Unf": 1, "LwQ": 2, "Rec": 3, "BLQ": 4, "ALQ": 5, "GLQ": 6},
    "GarageFinish": {"None": 0, "Unf": 1, "RFn": 2, "Fin": 3},
    "Functional": {"Sal": 0, "Sev": 1, "Maj2": 2, "Maj1": 3, "Mod": 4, "Min2": 5, "Min1": 6, "Typ": 7},
    "LotShape": {"IR3": 0, "IR2": 1, "IR1": 2, "Reg": 3},
    "LandSlope": {"Sev": 0, "Mod": 1, "Gtl": 2},
    "PavedDrive": {"N": 0, "P": 1, "Y": 2},
    "Street": {"Grvl": 0, "Pave": 1},
    "Alley": {"None": 0, "Grvl": 1, "Pave": 2},
    "CentralAir": {"N": 0, "Y": 1},
    "Utilities": {"ELO": 0, "NoSeWa": 1, "NoSewr": 2, "AllPub": 3},
}

y = train["SalePrice"]
y_log = np.log1p(y)
X = train.drop(columns=["Id", "SalePrice"])

X_train, X_val, y_train, y_val = train_test_split(
    X, y_log, test_size=0.2, random_state=42,
)

outlier_mask = (X_train["GrLivArea"] > 4000) & (y_train < np.log1p(200000))
X_train = X_train[~outlier_mask].reset_index(drop=True)
y_train = y_train[~outlier_mask].reset_index(drop=True)

for col in none_cols:
    X_train[col] = X_train[col].fillna("None")
for col in zero_cols:
    X_train[col] = X_train[col].fillna(0)

lot_frontage_medians = X_train.groupby("Neighborhood")["LotFrontage"].median()
global_lot_median = X_train["LotFrontage"].median()
X_train["LotFrontage"] = X_train["LotFrontage"].fillna(
    X_train["Neighborhood"].map(lot_frontage_medians)
)
X_train["LotFrontage"] = X_train["LotFrontage"].fillna(global_lot_median)

electrical_mode = X_train["Electrical"].mode()[0]
X_train["Electrical"] = X_train["Electrical"].fillna(electrical_mode)

for col, mapping in ordinal_maps.items():
    X_train[col] = X_train[col].map(mapping)

nominal_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
ohe.fit(X_train[nominal_cols])

train_ohe = pd.DataFrame(
    ohe.transform(X_train[nominal_cols]),
    columns=ohe.get_feature_names_out(nominal_cols),
    index=X_train.index,
)
X_train = pd.concat([X_train.drop(columns=nominal_cols), train_ohe], axis=1)

X_train["TotalSF"] = X_train["TotalBsmtSF"] + X_train["1stFlrSF"] + X_train["2ndFlrSF"]
X_train["TotalBathrooms"] = (
    X_train["FullBath"] + 0.5 * X_train["HalfBath"]
    + X_train["BsmtFullBath"] + 0.5 * X_train["BsmtHalfBath"]
)
X_train["TotalPorchSF"] = (
    X_train["OpenPorchSF"] + X_train["EnclosedPorch"]
    + X_train["3SsnPorch"] + X_train["ScreenPorch"] + X_train["WoodDeckSF"]
)
X_train["TotalFinishedBsmtSF"] = X_train["BsmtFinSF1"] + X_train["BsmtFinSF2"]
X_train["HouseAge"] = X_train["YrSold"] - X_train["YearBuilt"]
X_train["YearsSinceRemodel"] = X_train["YrSold"] - X_train["YearRemodAdd"]
X_train["GarageAge"] = np.where(
    X_train["GarageYrBlt"] > 0, X_train["YrSold"] - X_train["GarageYrBlt"], 0,
)
X_train["IsRemodeled"] = (X_train["YearRemodAdd"] != X_train["YearBuilt"]).astype(int)
X_train["LivingAreaPerRoom"] = X_train["GrLivArea"] / X_train["TotRmsAbvGrd"]

train_columns = X_train.columns.tolist()

corr_with_target = X_train.corrwith(y_train).abs().sort_values(ascending=False)
best_features = corr_with_target[corr_with_target > 0.05].index.tolist()

print(f"X_train shape: {X_train.shape}")
print(f"Number of best_features: {len(best_features)}")

X_train shape: (1166, 231)
Number of best_features: 158


In [15]:
test_processed = test.drop(columns=["Id"]).copy()
test_ids = test["Id"].copy()

for col in none_cols:
    test_processed[col] = test_processed[col].fillna("None")
for col in zero_cols:
    test_processed[col] = test_processed[col].fillna(0)

test_processed["LotFrontage"] = test_processed["LotFrontage"].fillna(
    test_processed["Neighborhood"].map(lot_frontage_medians)
)
test_processed["LotFrontage"] = test_processed["LotFrontage"].fillna(global_lot_median)
test_processed["Electrical"] = test_processed["Electrical"].fillna(electrical_mode)

extra_cat_fills = {
    "MSZoning":    train["MSZoning"].mode()[0],
    "Utilities":   train["Utilities"].mode()[0],
    "Functional":  train["Functional"].mode()[0],
    "KitchenQual": train["KitchenQual"].mode()[0],
    "SaleType":    train["SaleType"].mode()[0],
    "Exterior1st": train["Exterior1st"].mode()[0],
    "Exterior2nd": train["Exterior2nd"].mode()[0],
}
for col, val in extra_cat_fills.items():
    test_processed[col] = test_processed[col].fillna(val)

extra_num_zero = [
    "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF",
    "BsmtFullBath", "BsmtHalfBath", "GarageCars", "GarageArea",
]
for col in extra_num_zero:
    test_processed[col] = test_processed[col].fillna(0)

for col, mapping in ordinal_maps.items():
    test_processed[col] = test_processed[col].map(mapping)
    test_processed[col] = test_processed[col].fillna(0)

test_ohe = pd.DataFrame(
    ohe.transform(test_processed[nominal_cols]),
    columns=ohe.get_feature_names_out(nominal_cols),
    index=test_processed.index,
)
test_processed = pd.concat([test_processed.drop(columns=nominal_cols), test_ohe], axis=1)

test_processed["TotalSF"] = test_processed["TotalBsmtSF"] + test_processed["1stFlrSF"] + test_processed["2ndFlrSF"]
test_processed["TotalBathrooms"] = (
    test_processed["FullBath"] + 0.5 * test_processed["HalfBath"]
    + test_processed["BsmtFullBath"] + 0.5 * test_processed["BsmtHalfBath"]
)
test_processed["TotalPorchSF"] = (
    test_processed["OpenPorchSF"] + test_processed["EnclosedPorch"]
    + test_processed["3SsnPorch"] + test_processed["ScreenPorch"] + test_processed["WoodDeckSF"]
)
test_processed["TotalFinishedBsmtSF"] = test_processed["BsmtFinSF1"] + test_processed["BsmtFinSF2"]
test_processed["HouseAge"] = test_processed["YrSold"] - test_processed["YearBuilt"]
test_processed["YearsSinceRemodel"] = test_processed["YrSold"] - test_processed["YearRemodAdd"]
test_processed["GarageAge"] = np.where(
    test_processed["GarageYrBlt"] > 0,
    test_processed["YrSold"] - test_processed["GarageYrBlt"],
    0,
)
test_processed["IsRemodeled"] = (test_processed["YearRemodAdd"] != test_processed["YearBuilt"]).astype(int)
test_processed["LivingAreaPerRoom"] = test_processed["GrLivArea"] / test_processed["TotRmsAbvGrd"]

test_processed = test_processed[train_columns]

print(f"test_processed shape: {test_processed.shape}")
print(f"Remaining NaN: {test_processed.isnull().sum().sum()}")

test_processed shape: (1459, 231)
Remaining NaN: 0


In [16]:
loaded_model = mlflow.sklearn.load_model("models:/house-prices-best-model/latest")
print("Model loaded from registry.")

X_test_final = test_processed[best_features]
log_preds = loaded_model.predict(X_test_final)
price_preds = np.expm1(log_preds)

print(f"Prediction range: ${price_preds.min():.0f} - ${price_preds.max():.0f}")
print(f"Mean prediction: ${price_preds.mean():.0f}")

Model loaded from registry.
Prediction range: $40057 - $1643495
Mean prediction: $179055


In [17]:
submission = pd.DataFrame({
    "Id": test_ids,
    "SalePrice": price_preds,
})
submission.to_csv("submission.csv", index=False)

print(f"submission.csv written: {submission.shape}")
submission.head()

submission.csv written: (1459, 2)


,Id,SalePrice
0,1461,117767.592429
1,1462,156478.847318
2,1463,177608.901009
3,1464,189486.424738
4,1465,200658.278473
